# RAISE-Vision 1

In [2]:
#RUN THIS
from opentrons import protocol_api
import socket
from opentrons.types import *
import opentrons.execute

protocol = opentrons.execute.get_protocol_api("2.19")
protocol.home()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [3]:
#RUN THIS
#SETUP LAYOUT

well = protocol.load_labware("corning_24_wellplate_3.4ml_flat", 8) #corning_96_wellplate_360ul_flat
dryer_tips = protocol.load_labware("opentrons_96_tiprack_20ul", 10)
tips = protocol.load_labware("opentrons_96_tiprack_20ul", 11)
left = protocol.load_instrument("p20_single_gen2", "left", tip_racks=[tips])
right = protocol.load_instrument('p1000_single_gen2', 'right') ##no tipracks for the right

#Drypad
drypad = protocol.load_labware("fgl_drypad_tiprack", 6)

stage = protocol.load_labware("home_2_wellplate_10ul", 1)
trash = protocol.load_labware("fgl_trash", 5)

In [11]:
protocol.home()

In [4]:
#RUN THIS
#ACTIVATE CAMERA
#FUNCTIONS & CLASSES
class Camera:
    def __init__(self, ip_address: str, port: int):
        self.client_socket = socket.create_connection((ip_address, port))
    
    def take_image(self, fname: str) -> None:
        msg: str = f"take-image,{fname}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
        
    def new_folder(self, dir_name: str) -> None:
        msg: str = f"new-folder,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def analyze_images(self, dir_name: str) -> None:
        msg: str = f"analyze-images,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def __enter__(self):
        return self
    
    def close(self):
        self.client_socket.close()
    
    def __exit__(self, exc_type, exc_val, traceback):
        self.client_socket.close()

def change_offset(x: float = 0.0, y: float = 0.0, z: float = 0.0) -> Location:
        loc = Location(Point(x, y, z), None)
        return loc.move(Point(x, y, z))

def change_drypad_loco(dp_loco, kx, ky) -> Location:
    return dp_loco.move(Point(31.94 *kx, -28.24 * ky, 0))

In [5]:
#RUN THIS
#FUNCTIONS for the metallic mushroom on the RT stage
    
def drySides4p():
    stg = stage.wells()[1]
    dryer = tips.wells()[0]
    left.pick_up_tip(dryer)
    
    #Slay Right Side
    stage.set_offset(x=9, y=0, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 65)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=6.5, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=9, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    #Slay Left Side
    stage.set_offset(x= -11.5, y=0, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-8.1, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-11.5, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    #Slay Front Side
    stage.set_offset(x=-1.2, y=-10.5, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-1.2, y=-6.7, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.2, y=-10.5, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
    #Slay Back Side
    stage.set_offset(x=-1.2, y=11, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-1.2, y=8, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.2, y=11, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
    #Drop Tip
    #Throw Away the Tip to Trash place
    throw = trash.wells()[5]
    left.drop_tip(throw)

def drySides4p_v2(rolledDryerLoc):
    stg = stage.wells()[1]
    dryer = dryer_tips.wells()[rolledDryerLoc] #pickup from defined well
    left.pick_up_tip(dryer)
    
    #Slay Right Side
    stage.set_offset(x=9, y=0, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 65)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=6.5, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=9, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    #Slay Left Side
    stage.set_offset(x= -11.5, y=0, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-8.1, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-11.5, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    #Slay Front Side
    stage.set_offset(x=-1.2, y=-10.5, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-1.2, y=-6.7, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.2, y=-10.5, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
    #Slay Back Side
    stage.set_offset(x=-1.2, y=11, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-1.2, y=8, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.2, y=11, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
    #Drop Tip
    #Throw Away the Tip to Trash place
    throw = trash.wells()[5]
    left.drop_tip(throw)
    
def dryDropletTiprack(wellNum): 
    loc = drypad.wells()[wellNum]
    left.pick_up_tip(loc)


    #Dry the actual droplet
    stage.set_offset(x=-1.27, y=0, z=0.00)
    stg = stage.wells()[1]
    left.move_to(stg.top(140), force_direct = False)
    left.move_to(stg.top(52), force_direct = False, speed = 40)
    left.move_to(stg.top(49.3), force_direct = False, speed = 2) #47.1
    protocol.delay(3)
    left.move_to(stg.top(130), force_direct = False)
    
    #Throw Away the Drypad
    left.drop_tip(loc)

def droplet(j, run_dir):
    stage.set_offset(x=-1.27, y=0, z=0.00)
    stg = stage.wells()[1]
    
    #Locations 
    locA = stg.top(53.75)
    locB = stg.top(51.75) #48.75

    
    #ASPIRATE
    left.pick_up_tip()
    
    #prelens image
    fname = run_dir + f"pre_lens_{j}.jpg"
    cam.take_image(fname)
    
    left.aspirate(5, well["A1"]) # Aspirate 5 uL

    # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    fname = run_dir + f"Image_{j}.jpg"
    cam.take_image(fname)
    protocol.delay(1)
    
    #Drop the tip in the TRASH
    left.drop_tip() 

In [6]:
#RUN THIS
# RAISE VISION, RAISE OT-2 CAMERA CONNECTION - 2
# from time import time
IP_ADDRESS = "192.168.0.111"

PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

In [ ]:
left.drop_tip()

In [ ]:
#Drop Tip
#Throw Away the Tip to Trash place
throw = trash.wells()[5]
left.drop_tip(throw)

In [ ]:
#Throw Away the Drypad
left.drop_tip(drypad.wells()[0])

In [10]:
protocol.home()

In [8]:
a_dry = 0
j_tip = 0

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 402, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 415, in read_labware_latch
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 154, in get_labware_latch_status
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: 

In [9]:
#RAISE-Vision 1 with 2 racks **** <3
run_dir = '../20260807-senofilcon A MAX-4p4m-RV-RT (2_test)/'
cam.new_folder(run_dir)
drySides4p_v2(a_dry)
a_dry += 1
if (a_dry >= 96):
    a_dry = 0


protocol.delay(49 + 120)    #2 mins = 49    (4 points)
# # #     protocol.delay(13 +u 180)  #2 mins = 14    (8 points)

for j in range(10):
    droplet(j_tip, run_dir)
    j_tip += 1
    if (j_tip >= 96):
        j_tip = 0
    dryDropletTiprack(j)

cam.analyze_images(run_dir)

left.drop_tip()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

KeyboardInterrupt: 

In [12]:
#RAISE-Vision 1 **** <3
run_dir = '../20260703-senofilcon A MAX (2030)-4p4m-RV-RT (4)/'
cam.new_folder(run_dir)
drySides4p()


protocol.delay(49 + 120)    #2 mins = 49    (4 points)
# # #     protocol.delay(13 +u 180)  #2 mins = 14    (8 points)

for j in range(10):
    droplet(j, run_dir) 
    dryDropletTiprack(j)

cam.analyze_images(run_dir)

left.drop_tip()

detach tip called with no tip


<InstrumentContext: p20_single_v2.2 in LEFT>

In [10]:
cam.close()
# left.drop_tip()

In [ ]:
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

run_dir = '../02600115-somofilcon-4p4m-RV-RT (1)/'
cam.new_folder(run_dir)


fname = run_dir + '9.jpg"
cam.take_image(fname)
# cam.analyze_image(fname)

cam.analyze_images(run_dir)

cam.close()

### OTHER

In [ ]:
# #OTHER
# #FUNCTIONS

# def drySides8p():
#     tips.set_offset(x=0, y=0, z=0)
#     stg = stage.wells()[1]
#     dryer = tips.wells()[0]
#     left.pick_up_tip(dryer)
    
#     #Slay Right Side
#     stage.set_offset(x=9.6, y=1.65, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 65)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=8, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=9.6, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
#     #Front-Right Side
#     stage.set_offset(x=7.3, y=-5, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=5.4, y=-3.5, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=7.3, y=-5, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Slay Front Side
#     stage.set_offset(x=0, y=-8.5, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=0, y=-6.2, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=0, y=-8.5, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Front-Left Side
#     stage.set_offset(x=-7.2, y=-5.3, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=-5.6, y=-3.8, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=-7.2, y=-5.3, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
#     #Slay Left Side
#     stage.set_offset(x= -10, y=1.65, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=-8.1, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=-10, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Back-Left Side
#     stage.set_offset(x=-8, y=7.5, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=-5.8, y=6, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=-8, y=7.5, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Slay Back Side
#     stage.set_offset(x=0, y=12, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=0, y=10.3, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=0, y=12, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Back-Right Side
#     stage.set_offset(x=7.6, y=8.3, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=5.8, y=6.2, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=7.6, y=8.3, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Drop Tip
#     #Throw Away the Tip to Trash place
#     throw = trash.wells()[5]
#     left.drop_tip(throw)
    
# def drySides4p():
#     tips.set_offset(x=0, y=0, z=0)
#     stg = stage.wells()[1]
#     dryer = tips.wells()[0]
#     left.pick_up_tip(dryer)
    
#     #Slay Right Side
#     stage.set_offset(x=9.6, y=1.65, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 65)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=8, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=9.6, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Slay Left Side
#     stage.set_offset(x= -10, y=1.65, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=-8.1, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=-10, y=1.65, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
#     #Slay Front Side
#     stage.set_offset(x=0, y=-8.5, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=0, y=-6.2, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=0, y=-8.5, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
#     #Slay Back Side
#     stage.set_offset(x=0, y=12, z=0)
#     left.move_to(stg.top(40), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 25)
    
#     stage.set_offset(x=0, y=10.3, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)
    
#     stage.set_offset(x=0, y=12, z=0)
#     left.move_to(stg.top(22), force_direct = False, speed = 15)
#     left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
#     #Drop Tip
#     #Throw Away the Tip to Trash place
#     throw = trash.wells()[5]
#     left.drop_tip(throw)
    
# def dryDropletTiprack(wellNum): 
#     loc = drypad.wells()[wellNum]
#     left.pick_up_tip(loc)


#     #Dry the actual droplet
#     stage.set_offset(x=-0.15, y=1.65, z=0.00)
#     stg = stage.wells()[1]
#     left.move_to(stg.top(140), force_direct = False)
#     left.move_to(stg.top(51), force_direct = False, speed = 40)
#     left.move_to(stg.top(46.1), force_direct = False, speed = 2) #47.6
#     protocol.delay(3)
#     left.move_to(stg.top(130), force_direct = False)
    
#     #Throw Away the Drypad
#     throw = trash.wells()[wellNum]
#     left.drop_tip(throw)


# def droplet(j):
#     stage.set_offset(x=-0.2, y=1.7, z=0.00)
#     stg = stage.wells()[1]
    
#     #Locations 
#     locA = stg.top(52)
#     locB = stg.top(48.75)

    
#     #ASPIRATE
#     left.pick_up_tip()
#     left.aspirate(5, well["D5"]) # Aspirate 5 uL

#     # CREATE DROPLET ON PIPETTE TIP     
#     left.move_to(locA, force_direct = False)
#     left.dispense(5, locA, rate = 1) # Dispense 5 uL
#     left.move_to(locB, force_direct = True, speed = 0.5)
    
#     #Safety Factor
#     left.move_to(locA, force_direct = True, speed = 2)
    
#     #VIEW DROPLET + TAKE IMAGE   
#     protocol.delay(5)
#     fname = f"../Image_{j}.jpg"
#     cam.take_image(fname)
#     protocol.delay(1)
    
#     #Drop the tip in the TRASH
#     left.drop_tip() 

In [ ]:
cam.close()

# # DCA Play

In [ ]:
#this is for play
    
def DCAdroplet(j):
    stage.set_offset(x=-0.2, y=1.7, z=0.00)
    stg = stage.wells()[1]
    
    #Locations 
    locA = stg.top(50)
    locB = stg.top(47.65)

    
    #ASPIRATE
    left.pick_up_tip()
    left.aspirate(5, well["A1"]) # Aspirate 5 uL

#     # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    left.dispense(1, locA, rate = 0.1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    left.dispense(4, locB, rate = 0.1) # Dispense 5 uL
    
    #left.aspirate(5, rate = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    
    #Drop the tip in the TRASH
    left.drop_tip() 

DCAdroplet(j) 

In [ ]:
# stage = protocol.load_labware("home_2_wellplate_10ul", 5)

stage.set_offset(x=1, y=-90, z=0.00)
stg = stage.wells()[1]
    
#Locations 
loc_cam = stg.top(100)


# CREATE DROPLET ON PIPETTE TIP     
left.move_to(loc_cam, force_direct = False, speed = 10)


In [ ]:
left.drop_tip()

In [ ]:
protocol.home()

# Raise Vision 2 :)

In [ ]:
protocol.home()

In [ ]:
# from opentrons import protocol_api
# import socket
# from opentrons.types import *
# import opentrons.execute

# protocol = opentrons.execute.get_protocol_api("2.19")
# protocol.home()

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

#Pipettes
left = protocol.load_instrument("p20_single_gen2", "left")
right = protocol.load_instrument('p1000_single_gen2', 'right') ##no tipracks for the right

#Liquids Handling
well = protocol.load_labware("corning_24_wellplate_3.4ml_flat", 1)
tips = protocol.load_labware("opentrons_96_tiprack_20ul", 3) 

#distance from mushroom to pipette in y is 55.3mm)
stage = protocol.load_labware('home_2_wellplate_10ul', 4) 
gear = protocol.load_labware('fgl_1_tiprack_1000ul',7)
mushroom = protocol.load_labware('fgl_mush', 10) 
lens = protocol.load_labware('fgl_lens', 11)
drypad = protocol.load_labware('fgl_rv1_drypad', 6)


In [ ]:
right.drop_tip()

In [ ]:
def gearPick(): 
    loc = gear.wells_by_name()['A1']
    right.pick_up_tip(loc)

def mushy(location, height):
    
    mushroom.set_offset(x=0.00, y=-(63.3), z=0.00)
    #mushroom floors locations
    top = 76
    fourth = top-25.35*3
    third = top-25.35*2
    second = top-25.35
    
    loc = mushroom.wells_by_name()[location]
    right.move_to(loc.top(height), force_direct = False, speed = 35)
    mushroom.set_offset(x=0.00, y=-50, z=0.00)
    right.move_to(loc.top(height), force_direct = False, speed = 1)
    mushroom.set_offset(x=0.00, y=-80, z=0.00)
    right.move_to(loc.top(height), force_direct = False, speed = 5)
    right.move_to(loc.top(150), force_direct = False, speed = 35)


def flp(side):
    #defining stuff
    heightRack = 116.75
    mushroom.set_offset(x=0, y=-41, z=0.00)
    locA = mushroom.wells_by_name()['A3']
    locB = mushroom.wells_by_name()['A4']
      
    if (side == 'down'):
        right.move_to(locA.top(heightRack+15),force_direct = False, speed = 20)
        right.move_to(locA.top(heightRack),force_direct = False, speed = 15)
        right.move_to(locB.top(heightRack),force_direct = False, speed = 15)
        right.move_to(locB.top(heightRack+15),force_direct = False, speed = 10)
        
    if (side == 'up'):
        mushroom.set_offset(x=0, y=-41, z=-1)
        right.move_to(locB.top(heightRack+15), force_direct = False, speed = 20)
        right.move_to(locB.top(heightRack),force_direct = False, speed = 25)
        mushroom.set_offset(x=-1, y=-41, z=-1)
        right.move_to(locA.top(heightRack),force_direct = False, speed = 25)
        right.move_to(locA.top(heightRack+15),force_direct = False, speed = 5)

def dip():
    lens.set_offset(x=0.00, y=-54.5, z=0)
    loc = lens.wells_by_name()['D1']
    right.move_to(loc.top(25),force_direct = False, speed = 20)
    right.move_to(loc.top(0.5),force_direct = False, speed = 10)
    protocol.delay(3)
    right.move_to(loc.top(25),force_direct = False, speed = 20)
    
    
def stgDrop():
    stg = stage.wells()[1]
    
    stage.set_offset(x=0.5, y=-55.3, z=0.00)
    right.move_to(stg.top(25), force_direct = False, speed = 35)
    
    stage.set_offset(x=0.5, y=-65.3, z=0.00)
    right.move_to(stg.top(25), force_direct = False, speed = 2)
    
def stgPick():
    stg = stage.wells()[1]
    
    stage.set_offset(x=0.5, y=-65.3, z=0.00)
    right.move_to(stg.top(25), force_direct = False, speed = 0.5)
    
    stage.set_offset(x=0.5, y=-54, z=0.00)
    right.move_to(stg.top(25), force_direct = False, speed = 10)
    right.move_to(stg.top(50), force_direct = False, speed = 10)
    

In [ ]:
gearPick()
mushy('A1', top)
flp('down')
dip()
flp('up')
stgDrop()
stgPick()

In [ ]:
right.drop_tip()
protocol.home()

In [ ]:

    


def bp(location, height):
    loc = blip.wells_by_name()[location]
    blip.set_offset(x=0.00, y=-(55), z=0)
    
    right.move_to(loc.top(height), force_direct = False, speed = 20)
    


In [ ]:
protocol.home()

# RAISE VISION - HEATED LENS DESIGN CODE

In [ ]:
from opentrons import protocol_api
import socket
from opentrons.types import *
import opentrons.execute

protocol = opentrons.execute.get_protocol_api("2.19")
protocol.home()

#SETUP LAYOUT
well = protocol.load_labware("corning_24_wellplate_3.4ml_flat", 8)
tips = protocol.load_labware("opentrons_96_tiprack_20ul", 11)
left = protocol.load_instrument("p20_single_gen2", "left", tip_racks=[tips])
right = protocol.load_instrument('p1000_single_gen2', 'right') ##no tipracks for the right

#Drypad
drypad = protocol.load_labware("fgl_drypad_tiprack", 6)
stage = protocol.load_labware("home_2_wellplate_10ul", 4)
trash = protocol.load_labware("fgl_trash", 9)

In [ ]:
#Loading the heating module
hs_mod = protocol.load_module(module_name="heaterShakerModuleV1", location="1")
hs_mod.set_and_wait_for_temperature(55)


In [ ]:
left.drop_tip()

In [ ]:


#ACTIVATE CAMERA
#FUNCTIONS & CLASSES
class Camera:
    def __init__(self, ip_address: str, port: int):
        self.client_socket = socket.create_connection((ip_address, port))
    
    def take_image(self, fname: str) -> None:
        msg: str = f"take-image,{fname}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def __enter__(self):
        return self
    
    def close(self):
        self.client_socket.close()
    
    def __exit__(self, exc_type, exc_val, traceback):
        self.client_socket.close()

def change_offset(x: float = 0.0, y: float = 0.0, z: float = 0.0) -> Location:
        loc = Location(Point(x, y, z), None)
        return loc.move(Point(x, y, z))

def change_drypad_loco(dp_loco, kx, ky) -> Location:
    return dp_loco.move(Point(31.94 *kx, -28.24 * ky, 0))


#FUNCTIONS
def drySides():
    tips.set_offset(x=0, y=0, z=0)
    stg = stage.wells()[0]
    dryer = tips.wells()[0]
    left.pick_up_tip(dryer)
    
    #Slay Right Side
    stage.set_offset(x=9.6, y=1.65, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 65)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=8.0, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=9.6, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    #Front-Right Side
    stage.set_offset(x=7.3, y=-5, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 35)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=5.4, y=-3.8, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=7.3, y=-5, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    #Slay Front Side
    stage.set_offset(x=0, y=-8.5, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 35)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=0, y=-6.0, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=0, y=-8.5, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    #Front-Left Side
    stage.set_offset(x=-7.0, y=-5.3, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 35)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=-6.4, y=-4.1, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-7, y=-5.3, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    
    #Slay Left Side
    stage.set_offset(x= -9.6, y=1.65, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 35)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=-8.3, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-9.6, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    #Back-Left Side
    stage.set_offset(x=-7.5, y=7.5, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 35)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=-5.8, y=6.2, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-7.5, y=7.5, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    #Slay Back Side
    stage.set_offset(x=0, y=11.3, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 35)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=0, y=10.3, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=0, y=11.3, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    #Back-Right Side
    stage.set_offset(x=7.3, y=8.2, z=0)
    left.move_to(stg.top(100), force_direct = False, speed = 35)
    left.move_to(stg.top(50), force_direct = False, speed = 25)
    
    stage.set_offset(x=5.8, y=6.2, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=7.3, y=8.2, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 15)
    left.move_to(stg.top(100), force_direct = False, speed = 25)
    
    #Drop Tip
    #Throw Away the Tip to Trash place
    throw = trash.wells()[5]
    left.drop_tip(throw)
    
    
def dryDropletTiprack(wellNum): 
    loc = drypad.wells()[wellNum]
    left.pick_up_tip(loc)

    #Dry the actual droplet
    stage.set_offset(x=-0.15, y=1.65, z=0.00)
    stg = stage.wells()[0]
    left.move_to(stg.top(140), force_direct = False)
    left.move_to(stg.top(80), force_direct = False, speed = 40)
    left.move_to(stg.top(71.6), force_direct = False, speed = 2) #47.6
    protocol.delay(3)
    left.move_to(stg.top(130), force_direct = False)
    
    #Throw Away the Drypad
    throw = trash.wells()[wellNum]
    left.drop_tip(throw)
 
    
def droplet(j):
    stage.set_offset(x=-1, y=1, z=0.00)
    stg = stage.wells()[0]
    
    #Locations 
    locA = stg.top(80)
    locB = stg.top(74)

    
    #ASPIRATE
    left.pick_up_tip()
    left.aspirate(5, well["A3"]) # Aspirate 5 uL

    # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    fname = f"../Image_{j}.jpg"
    cam.take_image(fname)
    protocol.delay(1)
    
    #Drop the tip in the TRASH
    left.drop_tip() 

In [ ]:


def drySidesFaster():
    tips.set_offset(x=0, y=0, z=0)
    stg = stage.wells()[0]
    dryer = tips.wells()[0]
    left.pick_up_tip(dryer)
    
    #Slay Right Side
    stage.set_offset(x=9.6, y=1.65, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 200)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=8.0, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=9.6, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    #Front-Right Side
    stage.set_offset(x=7.3, y=-5, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=5.4, y=-3.8, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=7.3, y=-5, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    #Slay Front Side
    stage.set_offset(x=0, y=-8.5, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=0, y=-6.0, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=0, y=-8.5, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    #Front-Left Side
    stage.set_offset(x=-7.0, y=-5.3, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=-6.4, y=-4.1, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=-7, y=-5.3, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    
    #Slay Left Side
    stage.set_offset(x= -9.6, y=1.65, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=-8.3, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=-9.6, y=1.65, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    #Back-Left Side
    stage.set_offset(x=-7.5, y=7.5, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=-5.8, y=6.2, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=-7.5, y=7.5, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    #Slay Back Side
    stage.set_offset(x=0, y=11.3, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=0, y=10.3, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=0, y=11.3, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    #Back-Right Side
    stage.set_offset(x=7.3, y=8.2, z=0)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    
    stage.set_offset(x=5.8, y=6.2, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 2)
    protocol.delay(1)
    
    stage.set_offset(x=7.3, y=8.2, z=0)
    left.move_to(stg.top(50), force_direct = False, speed = 45)
    left.move_to(stg.top(75), force_direct = False, speed = 45)
    
    #Drop Tip
    #Throw Away the Tip to Trash place
    throw = trash.wells()[5]
    left.drop_tip(throw)
    
def dropletOneMin(j):
    stage.set_offset(x=-1, y=1, z=0.00)
    stg = stage.wells()[0]
    
    #Locations 
    locA = stg.top(75)
    locB = stg.top(74)

    
    #ASPIRATE
    left.pick_up_tip()
    left.aspirate(5, well["A3"]
                 ) # Aspirate 5 uL

    # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 15)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    fname = f"../Image_{j}.jpg"
    cam.take_image(fname)
    protocol.delay(1)
    
    #Drop the tip in the TRASH
    left.drop_tip() 


In [ ]:
#Have 30 seconds to place lens on the mushroom 
# RAISE VISION, RAISE OT-2 CAMERA CONNECTION - 2
from time import time
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

try:
    
    drySidesFaster() #takes 124.78 seconds (faster version takes 47 seconds instead)

    protocol.delay(40)
      
    fname = f"../pre_lens.jpg"
    cam.take_image(fname)

    for j in range(10):
        dropletOneMin(j) #Takes 40 seconds to take the picture, like 5 seconds to drop tip
#         t0 = time()
        dryDropletTiprack(j)#Dries the tip at 23 seconds, wait 60 seconds
#         t1 = time()
#         print(f'{t1-t0:.2f} sec')
        protocol.delay(2)
except:
    pass
finally:
    cam.close()
    left.drop_tip()

In [ ]:
left.drop_tip()